# SELF ATTENTION-MECHANISM WITH TRAINABLE WIEGHTS

We start with a set of input embeddings.

Each row represents one token, and each token is represented by a 3-dimensional embedding.

In this example, we have 5 input tokens, so the shape of `inputs` is:

`(5, 3)`

where:

* 5 = number of input tokens
* 3 = embedding dimension of each token

The numbers themselves are just example embedding values. In a real language model, these embeddings are learned during training.

For the rest of the example, we will focus on the second token, `starts`, and calculate its context vector.


In [32]:
import torch

inputs = torch.tensor([
    [0.55, 0.87, 0.66],  # journey
    [0.57, 0.85, 0.64],  # starts
    [0.22, 0.58, 0.33],  # with
    [0.77, 0.25, 0.10],  # one
    [0.05, 0.80, 0.55]   # step
])

### Now we will move forward by making 3 variables

In [33]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2


### Now we will initiliase it by 3 weight matrics 

#### These matrices are key query and value matrices 

### Initializing the Query, Key, and Value weight matrices

Self-attention uses three different weight matrices:

* `W_query` → used to create Query vectors
* `W_key` → used to create Key vectors
* `W_value` → used to create Value vectors

Each matrix has shape:

`(d_in, d_out) = (3, 2)`

These matrices transform the original 3-dimensional input embeddings into 2-dimensional Query, Key, and Value vectors.

The projections are:

`Q = XW_Q`

`K = XW_K`

`V = XW_V`

The weight matrices are parameters of the attention mechanism. During actual model training, their values are updated using gradient descent.

In this notebook, `requires_grad=False` is used only to keep the example output simple and focus on understanding the forward computation.


In [34]:
torch.manual_seed(100)
# Note that we are setting requires_grad=False to reduce clutter in the outputs for illustration purposes.
# If we were to use the weight matrices for model training, we would set requires_grad=True to update these matrices during model training.
w_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad= False)
w_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad= False)
w_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad= False)

In [35]:
print(w_key)

Parameter containing:
tensor([[0.1117, 0.8158],
        [0.2626, 0.4839],
        [0.6765, 0.7539]])


In [36]:
print(w_query)

Parameter containing:
tensor([[0.2627, 0.0428],
        [0.2080, 0.1180],
        [0.1217, 0.7356]])


In [37]:
print(w_value)

Parameter containing:
tensor([[0.7118, 0.7876],
        [0.4183, 0.9014],
        [0.9969, 0.7565]])


## Now we will compute the key, query and value matirce with the input

### Computing the Query, Key, and Value vectors

We now project the selected input vector `x_2` into three different representations.

For the second token:

`q_2 = x_2 W_Q`

`k_2 = x_2 W_K`

`v_2 = x_2 W_V`

Although all three vectors originate from the same input embedding, they represent different roles in the attention mechanism.

* Query asks: **What information am I looking for?**
* Key represents: **What information do I contain that can be matched?**
* Value represents: **What information should I actually contribute?**


In [38]:
query_2 = x_2 @ w_query    # Query vector for token 2
key_2 = x_2 @ w_key        # Key vector for token 2
value_2 = x_2 @ w_value    # Value vector for token 2

In [39]:
# As we can see based on the output for the query, this results in a 2-dimensional vector. 
# This is because: we set the number of columns of the corresponding weight matrix, via d_out, to 2:
print(query_2)

tensor([0.4044, 0.5955])


In [40]:
keys = inputs @ w_key
values = inputs @ w_value
queries = inputs @ w_query
print("keys.shape:", keys.shape)

print("values.shape:", values.shape)

print("queries.shape:", queries.shape)

keys.shape: torch.Size([5, 2])
values.shape: torch.Size([5, 2])
queries.shape: torch.Size([5, 2])


### Computing the attention scores

The attention score measures how strongly a Query matches a Key.

For the second token, we first calculate the score between:

`q_2` and `k_2`

using their dot product:

`ω_22 = q_2 · k_2`

A larger dot product means that the Query and Key are more aligned.

The notation `ω_ij` means:

* `i` = Query token
* `j` = Key token

So `ω_22` means the attention score between the Query of token 2 and the Key of token 2.


### Now we will compute our first attention score

In [41]:
keys_2 = keys[1] # A: Key vector of the second token
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.1003)


In [42]:
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.1121, 1.1003, 0.5840, 0.5797, 0.7395])


In [43]:
attn_scores = queries @ keys.T # omega
print(attn_scores)

tensor([[1.1352, 1.1233, 0.5960, 0.5934, 0.7539],
        [1.1121, 1.1003, 0.5840, 0.5797, 0.7395],
        [0.5994, 0.5930, 0.3148, 0.3123, 0.3986],
        [0.3822, 0.3767, 0.2031, 0.1706, 0.2712],
        [0.8667, 0.8584, 0.4539, 0.4673, 0.5671]])


In [44]:
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)
print(d_k)

tensor([0.2418, 0.2399, 0.1665, 0.1660, 0.1858])
2


### Converting attention scores into attention weights

The raw attention scores are converted into probabilities using softmax.

Before applying softmax, we divide the scores by:

`sqrt(d_k)`

where `d_k` is the dimensionality of the Key vectors.

The complete operation is:

`AttentionWeights = softmax(QK^T / sqrt(d_k))`

Softmax converts the scores into values between 0 and 1, and the values in each row sum to 1.

Therefore, the resulting values can be interpreted as attention weights.

For our example:

`d_k = 2`

because every Key vector has two dimensions.


In [45]:
import torch

# Define the tensor
tensor = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])

# Apply softmax without scaling
softmax_result = torch.softmax(tensor, dim=-1)
print("Softmax without scaling:", softmax_result)

# Multiply the tensor by 8 and then apply softmax
scaled_tensor = tensor * 8
softmax_scaled_result = torch.softmax(scaled_tensor, dim=-1)
print("Softmax after scaling (tensor * 8):", softmax_scaled_result)

Softmax without scaling: tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])
Softmax after scaling (tensor * 8): tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])


## Why do we divide by √d_k?

There are two important reasons.

### 1. Preventing extremely large values

The softmax function is sensitive to the magnitude of its inputs.

If attention scores become very large, softmax can become extremely "peaky".

For example, one value may receive almost all the probability while the remaining values become close to zero.

This can make gradients very small for the less-attended positions and can make optimization more difficult.

Scaling the scores helps keep their magnitude under control.

### 2. Controlling the variance of the dot product

Suppose the components of Query and Key are independent random variables with approximately zero mean and unit variance.

The dot product is:

`q · k = q_1k_1 + q_2k_2 + ... + q_dk_d`

As the dimension `d_k` increases, the variance of this sum increases approximately in proportion to `d_k`.

Therefore:

`Var(q · k) ≈ d_k`

Dividing by:

`sqrt(d_k)`

reduces the variance:

`Var((q · k) / sqrt(d_k)) ≈ 1`

This keeps the attention scores at a more stable scale before applying softmax.

This scaling factor is part of the standard scaled dot-product attention mechanism introduced with the Transformer architecture.


In [46]:
import numpy as np

# Function to compute variance before and after scaling
def compute_variance(dim, num_trials=1000):
    dot_products = []
    scaled_dot_products = []

    # Generate multiple random vectors and compute dot products
    for _ in range(num_trials):
        q = np.random.randn(dim)
        k = np.random.randn(dim)
        
        # Compute dot product
        dot_product = np.dot(q, k)
        dot_products.append(dot_product)
        
        # Scale the dot product by sqrt(dim)
        scaled_dot_product = dot_product / np.sqrt(dim)
        scaled_dot_products.append(scaled_dot_product)
    
    # Calculate variance of the dot products
    variance_before_scaling = np.var(dot_products)
    variance_after_scaling = np.var(scaled_dot_products)

    return variance_before_scaling, variance_after_scaling

# For dimension 5
variance_before_5, variance_after_5 = compute_variance(5)
print(f"Variance before scaling (dim=5): {variance_before_5}")
print(f"Variance after scaling (dim=5): {variance_after_5}")

# For dimension 20
variance_before_100, variance_after_100 = compute_variance(100)
print(f"Variance before scaling (dim=100): {variance_before_100}")
print(f"Variance after scaling (dim=100): {variance_after_100}")



Variance before scaling (dim=5): 4.884183958495453
Variance after scaling (dim=5): 0.9768367916990905
Variance before scaling (dim=100): 93.28584737373198
Variance after scaling (dim=100): 0.9328584737373199


In [47]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([1.0942, 1.3495])


# IMPLEMENTING A COMPACT SELF ATTENTION PYTHON CLASS

In the previous sections, we have gone through a lot of steps to compute the self-attention outputs.

This was mainly done for illustration purposes so we could go through one step at a time.

In practice, with the LLM implementation in the next chapter in mind, it is helpful to organize this code into a Python class as follows:

In [48]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

In this PyTorch code, SelfAttention_v1 is a class derived from nn.Module, which is a fundamental building block of PyTorch models, which provides necessary functionalities for model layer creation and management.

The init method initializes trainable weight matrices (W_query, W_key, and W_value) for queries, keys, and values, each transforming the input dimension d_in to an output dimension d_out.

During the forward pass, using the forward method, we compute the attention scores (attn_scores) by multiplying queries and keys, normalizing these scores using softmax.

Finally, we create a context vector by weighting the values with these normalized attention scores.

In [49]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.3274, 0.8104],
        [0.3271, 0.8095],
        [0.3154, 0.7773],
        [0.3133, 0.7714],
        [0.3199, 0.7898]], grad_fn=<MmBackward0>)


Since inputs contains six embedding vectors, we get a matrix storing the six context vectors, as shown in the above result.

As a quick check, notice how the second row ([0.3061, 0.8210]) matches the contents of context_vec_2 in the previous section.

We can improve the SelfAttention_v1 implementation further by utilizing PyTorch's nn.Linear layers, which effectively perform matrix multiplication when the bias units are disabled.

Additionally, a significant advantage of using nn.Linear instead of manually implementing nn.Parameter(torch.rand(...)) is that nn.Linear has an optimized weight initialization scheme, contributing to more stable and effective model training.

In [50]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [51]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0716,  0.0810],
        [-0.0717,  0.0809],
        [-0.0734,  0.0776],
        [-0.0739,  0.0768],
        [-0.0726,  0.0791]], grad_fn=<MmBackward0>)


Note that SelfAttention_v1 and SelfAttention_v2 give different outputs because they use different initial weights for the weight matrices since nn.Linear uses a more sophisticated weight initialization scheme.